In [29]:
pwd

'c:\\Users\\avina\\OneDrive\\Desktop\\VLBA\\feature_repo\\feature_repo'

In [30]:
import feast
import pandas as pd

df= pd.read_parquet("data/airline_features.parquet")

In [41]:
df.head()

,satisfaction,age,flight_distance,seat_comfort,departure_arrival_time_convenient,food_and_drink,gate_location,inflight_wifi_service,inflight_entertainment,online_support,...,travel_type_business,travel_type_personal,class_business,class_eco,class_eco_plus,total_service_score,total_delay,is_long_flight,passenger_id,event_timestamp
0,1,0.743590,0.031155,0,0,0,2,2,4,2,...,0,1,0,1,0,0.254545,0.0,0,1,2026-05-17 16:34:27.035573
1,1,0.512821,0.349804,0,0,0,3,0,2,2,...,0,1,1,0,0,0.254545,1.0,0,2,2026-05-17 16:34:27.035573
2,1,0.102564,0.302565,0,0,0,3,2,0,2,...,0,1,0,1,0,0.254545,0.0,0,3,2026-05-17 16:34:27.035573
3,1,0.679487,0.083031,0,0,0,3,3,4,3,...,0,1,0,1,0,0.163636,0.0,0,4,2026-05-17 16:34:27.035573
4,1,0.807692,0.044052,0,0,0,3,4,3,4,...,0,1,0,1,0,0.290909,0.0,0,5,2026-05-17 16:34:27.035573


In [42]:
predictor_df=df.drop(columns=["satisfaction"])
target_df=df[["satisfaction", "passenger_id", "event_timestamp"]]


In [43]:
# saving predictor and target dataframes as parquet files
predictor_df.to_parquet("data/predictor_data.parquet", index=False)
target_df.to_parquet("data/target_data.parquet", index=False)


In [6]:
!feast version  

Feast SDK Version: "0.63.0"


In [7]:
!feast init feature_repo


Creating a new Feast repository in c:\Users\avina\OneDrive\Desktop\VLBA\feature_repo.



In [45]:
predictor=pd.read_parquet("data/target_data.parquet")
predictor.head()

,satisfaction,passenger_id,event_timestamp
0,1,1,2026-05-17 16:34:27.035573
1,1,2,2026-05-17 16:34:27.035573
2,1,3,2026-05-17 16:34:27.035573
3,1,4,2026-05-17 16:34:27.035573
4,1,5,2026-05-17 16:34:27.035573


In [9]:

cd feature_repo

c:\Users\avina\OneDrive\Desktop\VLBA\feature_repo


In [46]:
predictor=pd.read_parquet("data/target_data.parquet")
predictor.head()

,satisfaction,passenger_id,event_timestamp
0,1,1,2026-05-17 16:34:27.035573
1,1,2,2026-05-17 16:34:27.035573
2,1,3,2026-05-17 16:34:27.035573
3,1,4,2026-05-17 16:34:27.035573
4,1,5,2026-05-17 16:34:27.035573


In [11]:
cd feature_repo

c:\Users\avina\OneDrive\Desktop\VLBA\feature_repo\feature_repo


In [14]:
!feast apply

No project found in the repository. Using project name feature_repo defined in feature_store.yaml
Applying changes for project feature_repo
Updated project feature_repo
	description: A project for driver statistics -> 
Created entity passenger_id
Deleted entity driver
Created feature view predictor_features
Created feature view target_features
Deleted feature view driver_hourly_stats_fresh
Deleted feature view driver_hourly_stats
Deleted on demand feature view transformed_conv_rate
Deleted on demand feature view transformed_conv_rate_fresh
Deleted feature service driver_activity_v3
Deleted feature service driver_activity_v2
Deleted feature service driver_activity_v1

Created sqlite table feature_repo_predictor_features
Created sqlite table feature_repo_target_features
Deleted sqlite table feature_repo_driver_hourly_stats_fresh
Deleted sqlite table feature_repo_driver_hourly_stats



In [48]:
from feast import FeatureStore
from feast.infra.offline_stores.file_source import SavedDatasetFileStorage

# Initialize the feature store

entity_df=pd.read_parquet("data/target_data.parquet")
store = FeatureStore(repo_path=".")

training_df=store.get_historical_features(
    entity_df=entity_df,
    features=[
        "predictor_features:age",
        "predictor_features:flight_distance",
        "predictor_features:seat_comfort",
        "predictor_features:departure_arrival_time_convenient",
        "predictor_features:food_and_drink",
        "predictor_features:gate_location",
        "predictor_features:inflight_wifi_service",
        "predictor_features:inflight_entertainment",
        "predictor_features:online_support",
        "predictor_features:ease_of_online_booking",
        "predictor_features:onboard_service",
        "predictor_features:leg_room_service",
        "predictor_features:baggage_handling",
        "predictor_features:checkin_service",
        "predictor_features:cleanliness",
        "predictor_features:online_boarding",
        "predictor_features:departure_delay",
        "predictor_features:arrival_delay",
        "predictor_features:total_service_score",
        "predictor_features:total_delay",
        "predictor_features:is_long_flight",
        "predictor_features:customer_type_loyal",
        "predictor_features:customer_type_disloyal",
        "predictor_features:travel_type_business",
        "predictor_features:travel_type_personal",
        "predictor_features:class_business",
        "predictor_features:class_eco",
        "predictor_features:class_eco_plus",
    ]
)

# create saved dataset file source
dataset=store.create_saved_dataset(
    from_=training_df,
    name="training_dataset",
    storage=SavedDatasetFileStorage("data/training_dataset.parquet"),
)

c:\Users\avina\OneDrive\Desktop\feast-mlops\.venv\Lib\site-packages\feast\feature_store.py:1530: RuntimeWarning: Saving dataset is an experimental feature. This API is unstable and it could and most probably will be changed in the future. We do not guarantee that future changes will maintain backward compatibility.
  warnings.warn(


In [49]:
# loading training data
training_df=pd.read_parquet("data/training_dataset.parquet")
training_df.head()

,satisfaction,passenger_id,event_timestamp,age,flight_distance,seat_comfort,departure_arrival_time_convenient,food_and_drink,gate_location,inflight_wifi_service,...,total_service_score,total_delay,is_long_flight,customer_type_loyal,customer_type_disloyal,travel_type_business,travel_type_personal,class_business,class_eco,class_eco_plus
0,1,1,2026-05-17 16:34:27.035573+00:00,0.743590,0.031155,0,0,0,2,2,...,0.254545,0.000000,0,1,0,0,1,0,1,0
1,0,86595,2026-05-17 16:34:27.035573+00:00,0.128205,0.274018,3,3,3,3,3,...,0.472727,0.167742,0,1,0,1,0,0,0,1
2,0,86594,2026-05-17 16:34:27.035573+00:00,0.128205,0.304159,3,1,1,1,3,...,0.400000,0.516129,0,1,0,1,0,0,0,1
3,0,86593,2026-05-17 16:34:27.035573+00:00,0.897436,0.468628,4,2,2,2,3,...,0.436364,0.000000,0,1,0,1,0,1,0,0
4,0,86592,2026-05-17 16:34:27.035573+00:00,0.153846,0.184031,3,1,4,4,3,...,0.472727,0.141935,0,1,0,1,0,0,0,1


In [50]:
# train_test_split and model training
from feast import FeatureStore
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from joblib import dump

# Initialize the feature store
store = FeatureStore(repo_path=".")
# Load the training dataset
traning_df=store.get_saved_dataset("training_dataset").to_df()
training_df

c:\Users\avina\OneDrive\Desktop\feast-mlops\.venv\Lib\site-packages\feast\feature_store.py:1582: RuntimeWarning: Retrieving datasets is an experimental feature. This API is unstable and it could and most probably will be changed in the future. We do not guarantee that future changes will maintain backward compatibility.
  warnings.warn(


,satisfaction,passenger_id,event_timestamp,age,flight_distance,seat_comfort,departure_arrival_time_convenient,food_and_drink,gate_location,inflight_wifi_service,...,total_service_score,total_delay,is_long_flight,customer_type_loyal,customer_type_disloyal,travel_type_business,travel_type_personal,class_business,class_eco,class_eco_plus
0,1,1,2026-05-17 16:34:27.035573+00:00,0.743590,0.031155,0,0,0,2,2,...,0.254545,0.000000,0,1,0,0,1,0,1,0
1,0,86595,2026-05-17 16:34:27.035573+00:00,0.128205,0.274018,3,3,3,3,3,...,0.472727,0.167742,0,1,0,1,0,0,0,1
2,0,86594,2026-05-17 16:34:27.035573+00:00,0.128205,0.304159,3,1,1,1,3,...,0.400000,0.516129,0,1,0,1,0,0,0,1
3,0,86593,2026-05-17 16:34:27.035573+00:00,0.897436,0.468628,4,2,2,2,3,...,0.436364,0.000000,0,1,0,1,0,1,0,0
4,0,86592,2026-05-17 16:34:27.035573+00:00,0.153846,0.184031,3,1,4,4,3,...,0.472727,0.141935,0,1,0,1,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
129875,0,43287,2026-05-17 16:34:27.035573+00:00,0.128205,0.255470,1,1,1,4,4,...,0.381818,0.470968,0,0,1,1,0,0,0,1
129876,0,43286,2026-05-17 16:34:27.035573+00:00,0.371795,0.227503,1,1,1,4,5,...,0.472727,0.000000,0,0,1,1,0,0,1,0
129877,0,43285,2026-05-17 16:34:27.035573+00:00,0.217949,0.282858,1,1,1,4,1,...,0.490909,1.000000,0,0,1,1,0,0,1,0
129878,0,43298,2026-05-17 16:34:27.035573+00:00,0.935897,0.052311,1,1,1,5,2,...,0.381818,0.000000,0,0,1,1,0,1,0,0


In [52]:
# Rank features by importance (RF + permutation + mutual info)
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import mutual_info_classif
from sklearn.inspection import permutation_importance

# 1. Random Forest impurity importance
rf = RandomForestClassifier(n_estimators=200, max_depth=15, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
print(f"Reference RF accuracy (all features): {rf.score(X_test, y_test):.4f}")
rf_imp = pd.Series(rf.feature_importances_, index=X_train.columns)

# 2. Permutation importance (model-agnostic, on test set)
print("Computing permutation importance...")
perm = permutation_importance(rf, X_test, y_test, n_repeats=5, random_state=42, n_jobs=-1)
perm_imp = pd.Series(perm.importances_mean, index=X_train.columns)

# 3. Mutual information
print("Computing mutual information...")
mi = mutual_info_classif(X_train, y_train, random_state=42)
mi_imp = pd.Series(mi, index=X_train.columns)

# Combine into a ranking table (lower avg_rank = more important)
ranks = pd.DataFrame({
    "rf_importance": rf_imp,
    "permutation_importance": perm_imp,
    "mutual_info": mi_imp,
})
ranks["rf_rank"] = ranks["rf_importance"].rank(ascending=False).astype(int)
ranks["perm_rank"] = ranks["permutation_importance"].rank(ascending=False).astype(int)
ranks["mi_rank"] = ranks["mutual_info"].rank(ascending=False).astype(int)
ranks["avg_rank"] = ranks[["rf_rank", "perm_rank", "mi_rank"]].mean(axis=1)
ranks = ranks.sort_values("avg_rank")
ranks

Reference RF accuracy (all features): 0.9480
Computing permutation importance...
Computing mutual information...


,rf_importance,permutation_importance,mutual_info,rf_rank,perm_rank,mi_rank,avg_rank
inflight_entertainment,0.209334,0.065420,0.236378,1,2,1,1.333333
seat_comfort,0.136329,0.118097,0.141376,2,1,3,2.000000
ease_of_online_booking,0.071715,0.010564,0.110432,4,6,4,4.666667
online_support,0.056383,0.018941,0.098587,5,4,5,4.666667
online_boarding,0.033016,0.013440,0.064683,9,5,7,7.000000
leg_room_service,0.035406,0.008414,0.061365,7,9,8,8.000000
total_service_score,0.078192,0.001016,0.151133,3,22,2,9.000000
food_and_drink,0.045642,0.008648,0.038751,6,8,16,10.000000
onboard_service,0.033842,0.004078,0.070030,8,16,6,10.000000
baggage_handling,0.018539,0.010108,0.054417,17,7,10,11.333333


In [ ]:
# Accuracy vs. top-K features (find the smallest K with comparable accuracy)
ordered = ranks.index.tolist()
results = []
for k in [5, 10, 15, 20, len(ordered)]:
    cols = ordered[:k]
    m = RandomForestClassifier(n_estimators=200, max_depth=15, random_state=42, n_jobs=-1)
    m.fit(X_train[cols], y_train)
    acc = m.score(X_test[cols], y_test)
    results.append({"top_k": k, "accuracy": acc})
    print(f"top-{k:>2}: accuracy={acc:.4f}")

pd.DataFrame(results)


top- 5: accuracy=0.8981
top-10: accuracy=0.9239
top-15: accuracy=0.9364
top-20: accuracy=0.9493
top-28: accuracy=0.9481


,top_k,accuracy
0,5,0.898121
1,10,0.923930
2,15,0.936403
3,20,0.949276
4,28,0.948106


In [ ]:
# Select top 10 features and retrain LogisticRegression on them
top_features = ranks.head(10).index.tolist()
print("Top 10 features:", top_features)

X_train_top = X_train[top_features]
X_test_top = X_test[top_features]

model = LogisticRegression(max_iter=1000)
model.fit(X_train_top, y_train)
print(f"\nAccuracy with top 10 features: {model.score(X_test_top, y_test):.4f}")

dump(model, "model_logistic_regression.joblib")


Top 10 features: ['inflight_entertainment', 'seat_comfort', 'ease_of_online_booking', 'online_support', 'online_boarding', 'leg_room_service', 'total_service_score', 'food_and_drink', 'onboard_service', 'baggage_handling']

Accuracy with top 10 features: 0.7975


['model_logistic_regression.joblib']

In [55]:
# for random forest 
model = RandomForestClassifier(n_estimators=200, max_depth=15, random_state=42, n_jobs=-1)
model.fit(X_train_top, y_train)
print(f"\nAccuracy with top 10 features: {model.score(X_test_top, y_test):.4f}")

dump(model, "model_random_forest.joblib")



Accuracy with top 10 features: 0.9239


['model_random_forest.joblib']

In [56]:
# creating online feature store
store=FeatureStore(repo_path=".")

store.materialize_incremental(end_date=pd.Timestamp.now())

Materializing 2 feature views to 2026-05-27 10:26:04+00:00 into the sqlite online store.

predictor_features from 2026-05-24 08:26:04+00:00 to 2026-05-27 10:26:04+00:00:
target_features from 2026-05-24 08:26:05+00:00 to 2026-05-27 10:26:04+00:00:


### getting online feature for prediction

In [57]:
from feast import FeatureStore
import pandas as pd
import joblib

# getting feature store
store=FeatureStore(repo_path=".")

# defining our feature names
feast_features=[
    "predictor_features:age",
    "predictor_features:flight_distance",
    "predictor_features:seat_comfort",
    "predictor_features:departure_arrival_time_convenient",
    "predictor_features:food_and_drink",
    "predictor_features:gate_location",
    "predictor_features:inflight_wifi_service",
    "predictor_features:inflight_entertainment",
    "predictor_features:online_support",
    "predictor_features:ease_of_online_booking",
    "predictor_features:onboard_service",
    "predictor_features:leg_room_service",
    "predictor_features:baggage_handling",
    "predictor_features:checkin_service",
    "predictor_features:cleanliness",
    "predictor_features:online_boarding",
    "predictor_features:departure_delay",
    "predictor_features:arrival_delay",
    "predictor_features:total_service_score",
    "predictor_features:total_delay",
    "predictor_features:is_long_flight",
    "predictor_features:customer_type_loyal",
    "predictor_features:customer_type_disloyal",
    "predictor_features:travel_type_business",
    "predictor_features:travel_type_personal",
    "predictor_features:class_business",
    "predictor_features:class_eco",
    "predictor_features:class_eco_plus",
]

# getting latest feature
features=store.get_online_features(
    features=feast_features,
    entity_rows=[{"passenger_id": 129879},{"passenger_id": 129878}],
).to_df()

features.head()

,passenger_id,customer_type_loyal,class_eco_plus,travel_type_business,onboard_service,travel_type_personal,total_service_score,inflight_entertainment,food_and_drink,age,...,departure_delay,online_boarding,departure_arrival_time_convenient,gate_location,flight_distance,is_long_flight,ease_of_online_booking,total_delay,customer_type_disloyal,cleanliness
0,129879,None,None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
1,129878,None,None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None


In [58]:
# call predict
model=joblib.load("model_logistic_regression.joblib")
# Use the feature names the model was trained on
X = features[model.feature_names_in_]
predictions=model.predict(X)
print(predictions)
probabilities=model.predict_proba(X)[:,1]
print("Predicted probabilities of satisfaction:", probabilities)

ValueError: Input X contains NaN.
LogisticRegression does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values